In [26]:
import wandb
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Initialize a W&B run
run = wandb.init()


####### MODELS
# 'master_thesis_math_lm/gpt2-math/gpt2-math-model:v0' --------------------- (17.03.25) Pre-trained
# 'master_thesis_math_lm/gpt2-math/gpt2-math-sft-final:v0' ----------------- (20.03.25) Curriculum learning
# 'master_thesis_math_lm/gpt2-math-instruct/gpt2-math-model-finetuned:v4' -- (24.04.25) 1st draft instruction learning



# Use the specified artifact
artifact = run.use_artifact('master_thesis_math_lm/gpt2-math/gpt2-math-model:v0', type='model')

# Download the artifact and get the directory path
artifact_dir = artifact.download()

# Load the tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained(artifact_dir)
model = GPT2LMHeadModel.from_pretrained(artifact_dir)



wandb: Currently logged in as: jonathantiedchen (master_thesis_math_lm). Use `wandb login --relogin` to force relogin


wandb: Downloading large artifact gpt2-math-model:v0, 479.31MB. 8 files... 
wandb:   8 of 8 files downloaded.  
Done. 0:0:1.5


In [28]:
# Example: Generate text using the model
input_text = "When I buy 3 apple and eat one, how many apples do I have left?"
input_ids = tokenizer.encode(input_text, return_tensors='pt')

attention_mask = torch.ones_like(input_ids)

output = model.generate(
    input_ids,
    attention_mask=attention_mask,  # Add this parameter
    max_length=300,
    num_return_sequences=1,
    repetition_penalty = 1
)

# Decode and print the generated text
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


When I buy 3 apple and eat one, how many apples do I have left?

I have 3 apples and I eat one apple. How many apples do I have left?

• You can't have more than one apple. – user17762 Nov 28 '16 at 16:18
• @user17762 I don't think so. – user17762 Nov 28 '16 at 16:19
• @user17762 I think you mean the number of apples you have left. – user17762 Nov 28 '16 at 16:20
• @user17762 I think you mean the number of apples you have left. – user17762 Nov 28 '16 at 16:21
• @user17762 I think you mean the number of apples you have left. – user17762 Nov 28 '16 at 16:22

You can't have more than one apple.

You can have more than one apple.

You can have more than one apple.

You can have more than one apple.

You can have more than one apple.

You can have more than one apple.

You can have more than one apple.

You can have more than one apple.

You can have more than one apple.

You can have more than one apple.

You can have more than one apple.

You can have more than one apple


## Query plain GPT2

In [31]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model_and_tokenizer(model_name="gpt2"):
    """
    Load GPT-2 model and tokenizer from Hugging Face.
    
    Args:
        model_name (str): Model variant to use. Options include:
            "gpt2" (small, 124M params)
            "gpt2-medium" (355M params)
            "gpt2-large" (774M params)
            "gpt2-xl" (1.5B params)
    
    Returns:
        tuple: (model, tokenizer)
    """
    print(f"Loading {model_name}...")

    # Load pre-trained model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    # Set padding token to be the EOS token
    tokenizer.pad_token = tokenizer.eos_token
    
    # Move model to GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    print(f"Model loaded on {device}")
    return model, tokenizer, device

def generate_text(model, tokenizer, prompt, device, 
                  max_length=100, 
                  temperature=0.7,
                  top_p=0.9,
                  do_sample=True):
    """
    Generate text from a prompt using GPT-2.
    
    Args:
        model: The GPT-2 model
        tokenizer: The GPT-2 tokenizer
        prompt (str): The text prompt to continue from
        device: The torch device to use
        max_length (int): Maximum length of generated text (including prompt)
        temperature (float): Controls randomness. Lower is more deterministic
        top_p (float): Nucleus sampling parameter (0.0-1.0)
        do_sample (bool): Whether to use sampling or greedy decoding
        
    Returns:
        str: The generated text
    """
    # Encode prompt to token IDs and move to the proper device
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    # Generate text
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            early_stopping=True
        )
    
    # Decode the generated token IDs back to text
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    
    return generated_text


# Load model and tokenizer
model, tokenizer, device = load_model_and_tokenizer("gpt2")  # You can change to larger variants


Loading gpt2...
Model loaded on cuda


In [32]:
# Example prompt
prompt = "Please write me a short poem."

# Generate text
generated_text = generate_text(
    model, 
    tokenizer, 
    prompt, 
    device,
    max_length=150,
    temperature=0.8
)

# Display results
print("\nPrompt:")
print(prompt)
print("\nGenerated text:")
print(generated_text)
print("\n" + "-"*50)


Prompt:
Please write me a short poem.

Generated text:
Please write me a short poem. I'll let you know when it's finished.

Thank you for reading. Please consider supporting the site with a donation today. Thank you.

If you enjoy PWInsider.com you can check out the AD-FREE PWInsider Elite section, which features exclusive audio updates, news, our critically acclaimed podcasts, interviews and more, right now for THREE DAYS free by clicking here!

--------------------------------------------------


## Plot GPT2 trained with EOS

In [33]:
import wandb
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Initialize a W&B run
api = wandb.Api()


####### MODELS
# 'master_thesis_math_lm/gpt2-math/gpt2-math-model:v0' --------------------- (17.03.25) Pre-trained
# 'master_thesis_math_lm/gpt2-math/gpt2-math-sft-final:v0' ----------------- (20.03.25) Curriculum learning
# 'master_thesis_math_lm/gpt2-math-instruct/gpt2-math-model-finetuned:v4' -- (24.04.25) 1st draft instruction learning



# Use the specified artifact
artifact = api.artifact('master_thesis_math_lm/gpt2-math-test/gpt2-math-model:v0', type='model')

# Download the artifact and get the directory path
artifact_dir = artifact.download()

# Load the tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained(artifact_dir)
model = GPT2LMHeadModel.from_pretrained(artifact_dir)


# Example: Generate text using the model
input_text = "Please write me a short poem."
input_ids = tokenizer.encode(input_text, return_tensors='pt')

attention_mask = torch.ones_like(input_ids)

output = model.generate(
    input_ids,
    attention_mask=attention_mask,  # Add this parameter
    max_length=100,
    num_return_sequences=1,
    repetition_penalty = 1
)

# Decode and print the generated text
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)



wandb: Downloading large artifact gpt2-math-model:v0, 479.31MB. 9 files... 
wandb:   9 of 9 files downloaded.  
Done. 0:0:1.0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Please write me a short poem.

# Theorem

Let $X$ be a field.

Then $X$ is a field.

Proof.

Let $f:X\to Y$ be a morphism.

Then $f$ is a morphism of $X$.

Then $f$ is a morphism of $Y$.

Then $f$ is a morphism of $X$.

Then $f$
